# PC-QAOA and PenaltyQAOA Comparison

This notebook reads the fixed COP manifest and saved results. It does not generate COPs or run optimization.

Mean plots show pointwise 95% Student-$t$ confidence intervals across COPs:

$$\bar y \pm t_{0.975,n-1}\frac{s}{\sqrt{n}}.$$

Here, $n$ is the number of COPs and $s$ is their sample standard deviation. Optimizer restarts and measurement shots are not independent observations. Paired comparisons use the per-COP difference between PC-QAOA and PenaltyQAOA. Intervals are omitted when fewer than two COPs contribute.

In [ ]:
from pathlib import Path
from collections import Counter
import json, os, shutil, subprocess, sys, warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import display, Image
from scipy.stats import t

candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "code-tuning"]
REPO = next(path for path in candidates if (path / "final_experiment/config.json").exists())
os.environ.setdefault("MPLCONFIGDIR", str(REPO / "final_experiment/.cache/matplotlib"))

MANIFEST_PATH = Path(os.environ.get(
    "PCQAOA_FINAL_MANIFEST", REPO / "final_experiment/artifacts/manifest.json"))
PC_DIR = Path(os.environ.get(
    "PCQAOA_FINAL_PC_RESULTS", REPO / "final_experiment/results/pcqaoa"))
PENALTY_DIR = Path(os.environ.get(
    "PCQAOA_FINAL_PENALTY_RESULTS", REPO / "final_experiment/results/penaltyqaoa"))
if not (PC_DIR / "run.json").exists():
    PC_DIR = REPO / "final_experiment/results/smoke_pcqaoa"
if not (PENALTY_DIR / "run.json").exists():
    PENALTY_DIR = REPO / "final_experiment/results/smoke_penaltyqaoa"
FIGURE_DIR = Path(os.environ.get(
    "PCQAOA_FINAL_FIGURES", REPO / "final_experiment/results/figures"))
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
VCG_DATABASE_PATH = Path(os.environ.get(
    "PCQAOA_FINAL_VCG_DATABASE", REPO / "final_experiment/vcg_results/selected_vcg_db.pkl"))
if not VCG_DATABASE_PATH.exists():
    VCG_DATABASE_PATH = REPO / "tuning/results/full/selected_vcg_db.pkl"
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "final_experiment"))
from resource_analysis import estimate_final_resources

COLORS = {"PC-QAOA": "#9063cd", "PenaltyQAOA": "#3e8fb0",
          "Disjoint": "#31748f", "Overlapping": "#eb6f92",
          "Teal": "#31748f", "Gold": "#f6c177", "Gray": "#6e6a86"}
MARKERS = {"PC-QAOA": "o", "PenaltyQAOA": "s"}

mpl.rcParams.update({
    "font.family": "serif", "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "cm", "font.size": 11, "axes.titlesize": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.22, "grid.linewidth": 0.5,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "savefig.facecolor": "white", "savefig.dpi": 300, "pdf.fonttype": 42,
    "pgf.texsystem": "pdflatex", "pgf.rcfonts": False, "text.usetex": False,
})

class FigureWriter:
    def __init__(self, directory, latex=True):
        self.directory = Path(directory)
        self.directory.mkdir(parents=True, exist_ok=True)
        self.latex = latex and bool(shutil.which("pdflatex"))
        self.font_status = ("LaTeX Computer Modern (PGF)" if self.latex else
                            "Matplotlib Computer Modern Math / DejaVu Serif")

    def save(self, figure, name):
        pdf = self.directory / f"{name}.pdf"
        png = self.directory / f"{name}.png"
        figure.tight_layout()
        if self.latex:
            try:
                figure.savefig(pdf, backend="pgf", bbox_inches="tight")
                if shutil.which("pdftocairo"):
                    subprocess.run(["pdftocairo", "-png", "-singlefile", "-r", "300",
                                    str(pdf), str(png.with_suffix(""))],
                                   check=True, capture_output=True)
                else:
                    figure.savefig(png, backend="pgf", bbox_inches="tight")
            except Exception as error:
                self.latex = False
                self.font_status = "Matplotlib Computer Modern Math / DejaVu Serif (LaTeX Failed)"
                warnings.warn(f"LaTeX rendering failed; using Matplotlib fonts: {error}")
        if not self.latex:
            figure.savefig(pdf, bbox_inches="tight")
            figure.savefig(png, bbox_inches="tight")
        (self.directory / "font_status.json").write_text(
            json.dumps({"font": self.font_status}, indent=2) + "\n")
        plt.close(figure)
        display(Image(filename=str(png), width=900))
        return png

writer = FigureWriter(FIGURE_DIR, latex=True)
print("Manifest:", MANIFEST_PATH)
print("PC-QAOA Results:", PC_DIR)
print("PenaltyQAOA Results:", PENALTY_DIR)

## Data Loading and Completion

The manifest defines the 500 COPs shared by both methods. Result files are joined by `instance_id`. A paired comparison includes a COP only when both methods have saved the required result.

In [ ]:
def load_json(path, default=None):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else default

def distribution_metrics(distribution, reference):
    distribution = np.asarray(distribution, dtype=float)
    total = distribution.sum()
    if total <= 0:
        return {"ar_feas": np.nan}
    distribution = distribution / total
    feasible = np.asarray(reference["feasible_mask"], dtype=bool)
    values = np.asarray(reference["objective_values"], dtype=float)
    p_feasible = float(distribution[feasible].sum())
    if p_feasible <= 0:
        return {"ar_feas": 0.0}
    feasible_objective = float(np.dot(distribution[feasible], values[feasible]) / p_feasible)
    denominator = float(reference["f_max_feasible"] - reference["f_star"])
    ar_feas = ((float(reference["f_max_feasible"]) - feasible_objective) / denominator
               if abs(denominator) > 1e-12 else np.nan)
    return {"ar_feas": ar_feas}

def load_results(directory, method, references):
    rows, depths = [], []
    directory = Path(directory)
    for path in sorted(directory.glob("final_*/result.json")):
        rows.append(load_json(path))
    for path in sorted(directory.glob("final_*/depths.json")):
        instance_id = path.parent.name
        reference = references[instance_id]
        metadata = {
            "n_x": reference["n_x"], "support": reference["support"],
            "regime": reference["regime"], "m": reference["m"], "o": reference["o"],
            "feasible_fraction": reference["feasible_fraction"], "delta": reference["delta"],
            "manuscript_delta": 5 + 2 * abs(reference["f_min_unconstrained"]),
        }
        for row in load_json(path, []):
            sampled = distribution_metrics(row["counts"], reference)
            exact = distribution_metrics(row["probabilities"], reference)
            derived = {**sampled, **{f"exact_{key}": value for key, value in exact.items()}}
            depths.append({
                "instance_id": instance_id, "method": method, **metadata, **derived,
                **{key: value for key, value in row.items()
                   if key not in ["angles", "history", "restarts", "counts", "probabilities"]},
            })
    return pd.DataFrame(rows), pd.DataFrame(depths)

def completion(directory, expected=None):
    statuses = Counter()
    directory = Path(directory)
    selection = load_json(directory / "selection.json")
    if expected is None:
        expected = len(selection["instance_ids"]) if selection else 500
    for path in directory.glob("final_*/status.json"):
        statuses[load_json(path)["status"]] += 1
    completed = len(list(directory.glob("final_*/result.json")))
    statuses["pending"] = max(0, expected - sum(statuses.values()))
    return {"expected": expected, "completed_results": completed,
            "statuses": dict(statuses)}

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Missing {MANIFEST_PATH}. Generate or copy the local manifest before using this notebook.")
manifest = load_json(MANIFEST_PATH)
references = {row["id"]: row for row in manifest["instances"]}
instances = pd.DataFrame(manifest["instances"])
instances["manuscript_delta"] = 5 + 2 * instances["f_min_unconstrained"].abs()
pc, pc_depths = load_results(PC_DIR, "PC-QAOA", references)
penalty, penalty_depths = load_results(PENALTY_DIR, "PenaltyQAOA", references)
for frame, label in [(pc, "PC-QAOA"), (penalty, "PenaltyQAOA")]:
    if not frame.empty:
        frame["method_label"] = label
for frame in [pc_depths, penalty_depths]:
    if not frame.empty:
        frame["method_label"] = frame["method"]
results = pd.concat([pc, penalty], ignore_index=True) if len(pc) or len(penalty) else pd.DataFrame()
depths = pd.concat([pc_depths, penalty_depths], ignore_index=True) if len(pc_depths) or len(penalty_depths) else pd.DataFrame()

print("Manifest COPs:", len(instances))
print("PC-QAOA:", json.dumps(completion(PC_DIR), indent=2))
print("PenaltyQAOA:", json.dumps(completion(PENALTY_DIR), indent=2))
display(instances[["id", "vcg_id", "vcg_family", "support", "n_x", "m", "o",
                   "n_structural", "n_penalty", "delta"]].head())

## Dataset Composition

Each COP contains one trained $L_{\mathrm{fid}}$ VCG constraint and one or two exactly structurally enforceable constraints. The objective coefficients are integers from $-5$ through $5$. PenaltyQAOA uses

$$\delta_{\mathrm{range}}=U_Q-L_Q+1,$$

where $L_Q$ and $U_Q$ are the sums of the negative and positive QUBO coefficients, respectively.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 6.5))
instances["support"].value_counts().sort_index().plot.bar(
    ax=axes[0, 0], color=COLORS["PC-QAOA"])
axes[0, 0].set(xlabel="VCG Support Size", ylabel="COPs", title="COP Count vs. VCG Support Size")
instances["vcg_family"].value_counts().sort_index().plot.bar(
    ax=axes[0, 1], color=COLORS["PenaltyQAOA"])
axes[0, 1].set(xlabel="VCG Constraint Family", ylabel="COPs", title="COP Count vs. VCG Constraint Family")
axes[0, 1].tick_params(axis="x", labelrotation=25)
instances["n_x"].value_counts().sort_index().plot.bar(
    ax=axes[1, 0], color=COLORS["Teal"])
axes[1, 0].set(xlabel=r"Problem Size ($n$)", ylabel="COPs", title=r"COP Count vs. Problem Size ($n$)")
cell = instances.groupby(["m", "o"]).size().unstack(fill_value=0)
cell.plot.bar(ax=axes[1, 1], color=[COLORS["Gray"], COLORS["Gold"], COLORS["Overlapping"]])
axes[1, 1].set(xlabel="Constraints per COP", ylabel="COPs", title="COP Count vs. Constraints per COP")
axes[1, 1].legend(title="Support Overlaps", frameon=False)
writer.save(fig, "dataset_composition")

exact_counts = Counter(category for values in instances["exact_categories"] for category in values)
partitions = instances.groupby(["n_structural", "n_penalty"]).size().rename("COPs").reset_index()
display(pd.Series(exact_counts, name="Occurrences").sort_index().to_frame())
display(partitions.rename(columns={"n_structural": "Structurally Enforced",
                                   "n_penalty": "Penalized"}))

## Optimization Metrics

For the feasible set $\mathcal{F}$ and the set of optimal feasible strings $\mathcal X^\star$,

$$
P(\mathrm{feas})=\sum_{x\in\mathcal{F}}P(x),\qquad
P(\mathrm{opt})=\sum_{x\in\mathcal X^\star}P(x).
$$

Slack registers are marginalized before these decision-level probabilities are calculated. As defined in the manuscript, the feasible-state objective and feasible approximation ratio are

$$
\bar f_{\mathrm{feas}}=
\frac{\sum_{x\in\mathcal{F}}f(x)P(x)}{P(\mathrm{feas})},\qquad
\mathrm{AR}_{\mathrm{feas}}
=\frac{\bar f_{\mathrm{feas}}-f_{\max}^{\mathcal{F}}}
{f_{\min}^{\mathcal{F}}-f_{\max}^{\mathcal{F}}}.
$$

The notebook assigns $\mathrm{AR}_{\mathrm{feas}}=0$ when $P(\mathrm{feas})=0$. Sampled metrics use 10,000 shots. Exact statevector probabilities are saved as diagnostics.

In [ ]:
def mean_ci(frame, group, value):
    stats = frame.groupby(group)[value].agg(["mean", "std", "count"])
    stats["sem"] = stats["std"] / np.sqrt(stats["count"])
    stats["ci95"] = t.ppf(0.975, stats["count"] - 1) * stats["sem"]
    stats.loc[stats["count"] < 2, "ci95"] = np.nan
    return stats.reset_index()

def depth_plot(frame, value, ylabel, title, name, reference=None):
    fig, ax = plt.subplots(figsize=(6.8, 4.0))
    for method, subset in frame.groupby("method_label", sort=False):
        stats = mean_ci(subset, "depth", value)
        ax.errorbar(stats["depth"], stats["mean"], yerr=stats["ci95"].fillna(0),
                    color=COLORS[method], marker=MARKERS[method], linewidth=1.5,
                    capsize=3, label=method)
    if reference is not None:
        ax.axhline(reference, color=COLORS["Gray"], linestyle="--", linewidth=1)
    ax.set(xlabel=r"Depth ($p$)", ylabel=ylabel, title=title)
    if len(frame):
        ax.set_xticks(sorted(frame["depth"].unique()))
    ax.legend(frameon=False)
    writer.save(fig, name)

if depths.empty:
    print("No saved depth results are available.")
else:
    depth_plot(depths, "p_optimal", r"$P(\mathrm{opt})$",
               r"$P(\mathrm{opt})$ vs. Depth ($p$)", "p_optimal_by_depth")
    depth_plot(depths, "p_feasible", r"$P(\mathrm{feas})$",
               r"$P(\mathrm{feas})$ vs. Depth ($p$)", "p_feasible_by_depth")
    depth_plot(depths, "ar_feas", r"$\mathrm{AR}_{\mathrm{feas}}$",
               r"$\mathrm{AR}_{\mathrm{feas}}$ vs. Depth ($p$)", "ar_feas_by_depth")
    display(mean_ci(depths, ["method_label", "depth"], "p_optimal"))

## Metrics by Problem Size and Constraint Structure

The size plots show $p=1$ and $p=5$ for both methods. The overlap plot uses $o=0$ for disjoint supports and $o\in\{1,2\}$ for one or two support overlaps. All intervals are pointwise 95% Student-$t$ intervals across COPs.

In [ ]:
target_depths = [depth for depth in [1, 5] if depth in set(depths.get("depth", []))]
if depths.empty or not target_depths:
    print("No depth-one or depth-five results are available.")
else:
    selected = depths[depths["depth"].isin(target_depths)].copy()
    metrics = [
        ("ar_feas", r"$\mathrm{AR}_{\mathrm{feas}}$"),
        ("p_feasible", r"$P(\mathrm{feas})$"),
        ("p_optimal", r"$P(\mathrm{opt})$"),
    ]
    line_styles = {1: "--", 5: "-"}
    depth_markers = {1: "^", 5: "o"}
    fig, axes = plt.subplots(1, 3, figsize=(13.2, 3.8))
    for ax, (metric, label) in zip(axes, metrics):
        for (method, depth), subset in selected.groupby(["method_label", "depth"], sort=False):
            stats = mean_ci(subset, "n_x", metric)
            ax.errorbar(stats["n_x"], stats["mean"], yerr=stats["ci95"].fillna(0),
                        color=COLORS[method], marker=depth_markers[int(depth)],
                        linestyle=line_styles[int(depth)], linewidth=1.5, capsize=3,
                        label=fr"{method}, $p={int(depth)}$")
        ax.set(xlabel=r"Problem Size ($n$)", ylabel=label,
               title=fr"{label} vs. Problem Size ($n$)")
        ax.set_xticks(sorted(instances["n_x"].unique()))
        ax.legend(frameon=False, fontsize=8)
    writer.save(fig, "manuscript_metrics_by_size")

    final_depth = 5 if 5 in target_depths else max(target_depths)
    final = selected[selected["depth"] == final_depth].copy()
    overlap_labels = {0: r"Disjoint ($o=0$)", 1: r"Overlap Level $o=1$",
                      2: r"Overlap Level $o=2$"}
    overlap_styles = {0: "-", 1: "--", 2: ":"}
    overlap_markers = {0: "o", 1: "s", 2: "D"}
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0), sharey=True)
    for ax, method in zip(axes, ["PC-QAOA", "PenaltyQAOA"]):
        subset = final[final["method_label"] == method]
        for overlap, group in subset.groupby("o"):
            stats = mean_ci(group, "n_x", "p_optimal")
            ax.errorbar(stats["n_x"], stats["mean"], yerr=stats["ci95"].fillna(0),
                        color=COLORS[method], linestyle=overlap_styles[int(overlap)],
                        marker=overlap_markers[int(overlap)], linewidth=1.5, capsize=3,
                        label=overlap_labels[int(overlap)])
        ax.set(xlabel=r"Problem Size ($n$)", ylabel=r"$P(\mathrm{opt})$",
               title=fr"{method}: $P(\mathrm{{opt}})$ vs. Problem Size ($n$)")
        ax.set_xticks(sorted(instances["n_x"].unique()))
        ax.legend(frameon=False, fontsize=8)
    fig.suptitle(fr"$P(\mathrm{{opt}})$ vs. Problem Size ($n$) by Overlap Level at $p={final_depth}$")
    writer.save(fig, "p_optimal_by_overlap_level")

    threshold = (depths.assign(reached=depths["p_feasible"] >= 0.75)
                 .groupby(["method_label", "depth"])["reached"]
                 .agg(["mean", "count"]).reset_index())
    fig, ax = plt.subplots(figsize=(6.8, 4.0))
    for method, subset in threshold.groupby("method_label", sort=False):
        ax.plot(subset["depth"], subset["mean"], color=COLORS[method],
                marker=MARKERS[method], linewidth=1.5, label=method)
    ax.set(xlabel=r"Depth ($p$)",
           ylabel=r"Fraction with $P(\mathrm{feas})\geq 0.75$",
           title=r"Threshold-Attainment Fraction vs. Depth ($p$)", ylim=(-0.02, 1.02))
    ax.set_xticks(sorted(depths["depth"].unique()))
    ax.legend(frameon=False)
    writer.save(fig, "feasibility_threshold_by_depth")

    display(final.groupby(["method_label", "regime"])
            [["ar_feas", "p_feasible", "p_optimal"]].agg(["mean", "std", "count"]))

## Paired Final-Depth Comparison

The table and difference plot compare the two methods on the same COPs. A positive difference means PC-QAOA has higher sampled $P(\mathrm{opt})$. Results are separated by decision size and by disjoint or overlapping constraint supports.

In [ ]:
if depths.empty:
    print("No paired depth results are available.")
else:
    final_depth = int(depths["depth"].max())
    final = depths[depths["depth"] == final_depth]
    paired = final.pivot(index="instance_id", columns="method_label", values="p_optimal")
    required = {"PC-QAOA", "PenaltyQAOA"}
    if not required.issubset(paired.columns):
        paired = pd.DataFrame()
    else:
        paired = paired[["PC-QAOA", "PenaltyQAOA"]].dropna()
        paired = paired.join(instances.set_index("id")[["n_x", "regime", "support", "m", "o"]])
    if paired.empty:
        print("No COP has results from both methods at the final available depth.")
    else:
        paired["Difference"] = paired["PC-QAOA"] - paired["PenaltyQAOA"]
        display(paired.describe())
        fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
        axes[0].scatter(paired["PenaltyQAOA"], paired["PC-QAOA"],
                        c=paired["n_x"], cmap="viridis", s=30, alpha=0.8)
        bound = max(paired[["PC-QAOA", "PenaltyQAOA"]].to_numpy().max(), 1e-3)
        axes[0].plot([0, bound], [0, bound], "--", color=COLORS["Gray"], linewidth=1)
        axes[0].set(xlabel=r"PenaltyQAOA $P(\mathrm{opt})$",
                    ylabel=r"PC-QAOA $P(\mathrm{opt})$",
                    title=fr"PC-QAOA $P(\mathrm{{opt}})$ vs. PenaltyQAOA $P(\mathrm{{opt}})$ at $p={final_depth}$")
        groups = [paired.loc[paired.regime == name, "Difference"]
                  for name in ["disjoint", "overlapping"]]
        means = [group.mean() for group in groups]
        errors = [t.ppf(0.975, len(group)-1) * group.std(ddof=1) / np.sqrt(len(group))
                  if len(group) > 1 else np.nan for group in groups]
        axes[1].bar(["Disjoint", "Overlapping"], means, yerr=np.nan_to_num(errors),
                    color=[COLORS["Disjoint"], COLORS["Overlapping"]], capsize=4)
        axes[1].axhline(0, color=COLORS["Gray"], linewidth=1)
        axes[1].set(ylabel=r"$\Delta P(\mathrm{opt})$",
                    title=r"Paired $\Delta P(\mathrm{opt})$ vs. Constraint Structure")
        writer.save(fig, "paired_final_depth")

        by_size = paired.groupby("n_x")[["PC-QAOA", "PenaltyQAOA"]].agg(["mean", "count"])
        display(by_size)

## Feasibility and Penalty Diagnostics

The first panel compares the selected coefficient-range penalty

$$
\delta_{\mathrm{range}}=U_Q-L_Q+1
$$

with the manuscript rule $\delta_{\mathrm{manuscript}}=5+2|f_{\min}|$. The other panels compare $P(\mathrm{feas})$ with the fraction of feasible decision strings for PC-QAOA and PenaltyQAOA at $p=5$. The dashed diagonal is the feasible probability produced by a uniform distribution over decision strings. Points above it assign more probability to feasible strings than uniform sampling.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.8))
delta_points = axes[0].scatter(
    instances["manuscript_delta"], instances["delta"],
    c=instances["n_x"], cmap="viridis", s=24, alpha=0.75)
fig.colorbar(delta_points, ax=axes[0], label=r"Problem Size ($n$)")
bound = float(max(instances["manuscript_delta"].max(), instances["delta"].max()))
axes[0].plot([0, bound], [0, bound], "--", color=COLORS["Gray"], linewidth=1)
axes[0].set(xlabel=r"Manuscript $\delta=5+2|f_{\min}|$",
            ylabel=r"Selected $\delta_{\mathrm{range}}$",
            title=r"$\delta_{\mathrm{range}}$ vs. Manuscript $\delta$")

final_depth = 5 if not depths.empty and (depths["depth"] == 5).any() else (
    int(depths["depth"].max()) if not depths.empty else None)
method_final = {
    method: (depths[(depths["method_label"] == method) &
                    (depths["depth"] == final_depth)].copy()
             if final_depth is not None else pd.DataFrame())
    for method in ["PC-QAOA", "PenaltyQAOA"]
}
for ax, method in zip(axes[1:], ["PC-QAOA", "PenaltyQAOA"]):
    subset = method_final[method]
    if subset.empty:
        ax.text(0.5, 0.5, f"No Completed $p={final_depth}$ Results",
                ha="center", va="center", transform=ax.transAxes)
    else:
        points = ax.scatter(
            subset["feasible_fraction"], subset["p_feasible"],
            c=subset["n_x"], cmap="viridis", s=30, alpha=0.8)
        fig.colorbar(points, ax=ax, label=r"Problem Size ($n$)")
        ax.plot([0, 1], [0, 1], "--", color=COLORS["Gray"], linewidth=1)
    ax.set(xlabel="Feasible Fraction", ylabel=r"$P(\mathrm{feas})$",
           title=fr"{method} $P(\mathrm{{feas}})$ vs. Feasible Fraction")
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
writer.save(fig, "feasibility_fraction_diagnostics")

penalty_final = method_final["PenaltyQAOA"]
if not penalty_final.empty:
    display(penalty_final[["instance_id", "n_x", "n_total", "delta", "manuscript_delta",
                           "feasible_fraction", "p_feasible", "p_optimal",
                           "ar_feas"]].describe().T)

## Circuit Resources and Runtime

The resource estimates use the fixed constraint partition and the saved $L_{\mathrm{fid}}$ VCGs for every COP. They count the circuits executed by the final experiment at $p=1$ and $p=5$.

- NISQ counts use $\{R_X,R_Y,R_Z,H,\mathrm{CNOT},X,Y,Z,T,S\}$.
- FTQC counts synthesize rotations into $\{H,T,\mathrm{CNOT},S\}$.
- Logical register qubits include decision and slack qubits. Total wires also include work qubits introduced by the gate decomposition.
- The counts do not include error correction, hardware connectivity, or SWAP routing.
- Curves show means and pointwise 95% Student-$t$ intervals across COPs.

In [ ]:
resource_frame = estimate_final_resources(
    MANIFEST_PATH, VCG_DATABASE_PATH,
    cache_dir=REPO / "final_experiment/results/resource_estimates",
    depths=(1, 5),
)
print("Resource Rows:", len(resource_frame))
display(resource_frame.groupby(["method", "gate_set", "depth"])[
    ["logical_qubits", "total_wires", "total_gates", "two_qubit_gates", "t_gates"]
].agg(["mean", "std", "count"]))

qubits = (resource_frame[(resource_frame["gate_set"] == "nisq") &
                          (resource_frame["depth"] == 1)]
          .drop_duplicates(["instance_id", "method"]))
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0))
for ax, metric, title in [
    (axes[0], "logical_qubits", r"Logical Register Qubits vs. Problem Size ($n$)"),
    (axes[1], "total_wires", r"Total Decomposition Wires vs. Problem Size ($n$)"),
]:
    for method, subset in qubits.groupby("method", sort=False):
        stats = mean_ci(subset, "n_x", metric)
        ax.errorbar(stats["n_x"], stats["mean"], yerr=stats["ci95"].fillna(0),
                    color=COLORS[method], marker=MARKERS[method], linewidth=1.5,
                    capsize=3, label=method)
    ax.set(xlabel=r"Problem Size ($n$)", ylabel="Qubits", title=title)
    ax.set_xticks(sorted(instances["n_x"].unique()))
    ax.legend(frameon=False)
writer.save(fig, "circuit_qubits")

resource_styles = {1: "--", 5: "-"}
resource_markers = {1: "^", 5: "o"}
for gate_set, metrics, filename in [
    ("nisq", [("total_gates", "Total Gates"),
              ("two_qubit_gates", "CNOT Gates")], "circuit_resources_nisq"),
    ("ftqc", [("total_gates", "Total Gates"),
              ("t_gates", "T Gates")], "circuit_resources_ftqc"),
]:
    subset = resource_frame[resource_frame["gate_set"] == gate_set]
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0))
    for ax, (metric, label) in zip(axes, metrics):
        for (method, depth), group in subset.groupby(["method", "depth"], sort=False):
            stats = mean_ci(group, "n_x", metric)
            ax.errorbar(stats["n_x"], stats["mean"], yerr=stats["ci95"].fillna(0),
                        color=COLORS[method], linestyle=resource_styles[int(depth)],
                        marker=resource_markers[int(depth)], linewidth=1.5, capsize=3,
                        label=fr"{method}, $p={int(depth)}$")
        ax.set(xlabel=r"Problem Size ($n$)", ylabel=label,
               title=fr"{label} vs. Problem Size ($n$) ({gate_set.upper()})")
        ax.set_xticks(sorted(instances["n_x"].unique()))
        ax.legend(frameon=False, fontsize=8)
    writer.save(fig, filename)

if not results.empty:
    fig, ax = plt.subplots(figsize=(6.8, 4.0))
    for method, subset in results.groupby("method_label", sort=False):
        ax.scatter(subset["n_total"], subset["optimize_seconds"],
                   color=COLORS[method], marker=MARKERS[method], s=28,
                   alpha=0.75, label=method)
    ax.set(xlabel="Simulated Qubits", ylabel="Optimization Time (s)",
           title="Optimization Time vs. Simulated Qubits")
    ax.legend(frameon=False)
    writer.save(fig, "optimization_runtime")
    display(results.groupby("method_label")[["compile_seconds", "optimize_seconds",
                                              "process_peak_rss_mb"]]
            .agg(["mean", "std", "count"]))

## Reproducibility

`run.json` records the method, configuration, environment, source hashes, VCG database hash, and dataset-manifest hash. `selection.json` records the COP IDs assigned to the run or shard. A resume is rejected when these identifiers do not match the saved run.

In [ ]:
for method, directory in [("PC-QAOA", PC_DIR), ("PenaltyQAOA", PENALTY_DIR)]:
    print(f"\n{method}")
    run = load_json(directory / "run.json")
    selection = load_json(directory / "selection.json")
    if run is None:
        print("No run metadata found.")
    else:
        print("Run Identifier:", run["identifier"])
        print("Manifest Hash:", run["manifest_hash"])
        print("Profile:", run["config"]["profile"])
        print("Selected COPs:", len(selection["instance_ids"]) if selection else "Unknown")
print("\nFigure Fonts:", writer.font_status)
print("Figure Directory:", FIGURE_DIR)